# Yearbook — exploratory data analysis

Frontal high-school yearbook portraits (1905–2013), labelled by gender (binary classification, scored on accuracy). The dataset is used to study temporal drift, so the views below focus on how the label balance and the portraits themselves change across a century.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from drift_happens.dataset.yearbook.const import YB_UNPACK_DIR
from drift_happens.dataset.yearbook.transform import load_downscaled_images_into_df

plt.rcParams["axes.unicode_minus"] = False
PRIMARY, SECONDARY = "#4C78A8", "#E45756"

yearbook = load_downscaled_images_into_df(
    YB_UNPACK_DIR / "faces_aligned_small_mirrored_co_aligned_cropped_cleaned"
)
yearbook.head()

## Dataset at a glance

In [ ]:
missing = yearbook[["gender", "year"]].isna().mean().max()
pd.Series({
    "Samples": f"{len(yearbook):,}",
    "Years": f"{yearbook['year'].min()}–{yearbook['year'].max()}",
    "Classes": f"{yearbook['gender'].nunique()} ({', '.join(sorted(yearbook['gender'].unique()))})",
    "Missing (gender, year)": f"{missing:.1%}",
    "Task": "binary classification",
    "Metric": "accuracy",
}, name="yearbook")

## Temporal coverage

In [ ]:
per_year = yearbook["year"].value_counts().sort_index()

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(per_year.index, per_year.values, color=PRIMARY)
ax.set_xlabel("Year")
ax.set_ylabel("Portraits")
ax.set_title("Portraits per year")
fig.tight_layout()

## Gender balance over time

The label proportions shift across decades, a source of drift the benchmark targets.

In [ ]:
by_year = yearbook.groupby("year")["gender"].value_counts().unstack(fill_value=0)
share = by_year.div(by_year.sum(axis=1), axis=0)

fig, ax = plt.subplots(figsize=(10, 4))
ax.stackplot(share.index, share["F"], share["M"], labels=["Female", "Male"],
             colors=[SECONDARY, PRIMARY])
ax.set_xlabel("Year")
ax.set_ylabel("Share")
ax.set_ylim(0, 1)
ax.set_title("Gender share per year")
ax.legend(loc="lower right")
fig.tight_layout()

## Sample portraits across the century

In [ ]:
decades = range(1910, 2011, 10)
samples = []
for start in decades:
    window = yearbook[(yearbook["year"] >= start) & (yearbook["year"] < start + 10)]
    if len(window):
        samples.append(window.iloc[0])

fig, axes = plt.subplots(2, 6, figsize=(12, 4.5))
for ax, row in zip(axes.ravel(), samples):
    portrait = np.asarray(row["img"], dtype=float)
    ax.imshow(portrait / 255 if portrait.max() > 1 else portrait)
    ax.set_title(f"{row['year']} · {row['gender']}", fontsize=9)
    ax.axis("off")
for ax in axes.ravel()[len(samples):]:
    ax.axis("off")
fig.suptitle("One portrait per decade")
fig.tight_layout()

## Mean portrait per decade

Averaging every portrait in a decade exposes how pose, framing, and image quality drift over time.

In [ ]:
def mean_image(images):
    stacked = np.stack([np.asarray(img, dtype=float) for img in images]).mean(axis=0)
    return stacked / 255 if stacked.max() > 1 else stacked

decades = list(range(1910, 2011, 10))
fig, axes = plt.subplots(2, 6, figsize=(12, 4.5))
for ax, start in zip(axes.ravel(), decades):
    window = yearbook[(yearbook["year"] >= start) & (yearbook["year"] < start + 10)]
    ax.imshow(mean_image(window["img"].to_list()))
    ax.set_title(f"{start}s · n={len(window)}", fontsize=8)
    ax.axis("off")
for ax in axes.ravel()[len(decades):]:
    ax.axis("off")
fig.suptitle("Mean portrait per decade")
fig.tight_layout()